# meeting_record_weekly_report — Meeting Record 每週進數與寄件人報告

每週跑一次，看 meeting record 兩張表這幾週**進了幾筆**、**是誰寄來的（誰記的）**、**信件標題**。只讀、不寫表，可隨時 Run All。

- 來源：`{catalog}.{schema}.b_internal_meeting_record`、`{catalog}.{schema}.b_internal_meeting_record_scm`
- 週次以 `create_date` 為基準、週一起算；範圍 = `run_date` 所在週（未完整，標 ongoing）+ 往回 `weeks_back` 個完整週
- 寄件人以 `mail_from` 內的 email（小寫）歸戶；抓不到 email 時用整串 `mail_from`

| cell | 內容 |
|---|---|
| [c10] | 讀兩張表（先依日期過濾，只取 4 個欄位） |
| [c11] | 每週筆數 + 寄件人數（沒資料的週補 0） |
| [c12] | 每週寄件人統計，文字報告印在 log |
| [c13] | 寄件人 × 週次 對照表：誰每週都有記、誰漏了 |
| [c14] | 每週筆數 bar chart |
| [c15] | 最近 `detail_weeks` 週的信件明細（寄件人、標題） |


In [ ]:
# [c01] params
# 只讀報表：不寫表，可以隨時重跑。預設值取自 config/project.yml。
# 注意：widget 建立過後改預設值不會生效；要重設請先 dbutils.widgets.removeAll() 再跑本 cell。
from datetime import date, datetime

dbutils.widgets.text("catalog", "micenter")
dbutils.widgets.text("schema", "mi3_datahub_prod")
dbutils.widgets.text("tables", "b_internal_meeting_record,b_internal_meeting_record_scm")
dbutils.widgets.text("run_date", "")         # yyyy-MM-dd；空 = 今天。這天所在的週 = 「本週」
dbutils.widgets.text("weeks_back", "4")      # 往回幾個完整週
dbutils.widgets.text("detail_weeks", "2")    # 明細列最近幾週（含本週）
dbutils.widgets.text("detail_limit", "500")  # 明細最多列幾筆
dbutils.widgets.text("top_senders", "20")    # 文字報告每週列前幾位寄件人

_run_date = dbutils.widgets.get("run_date").strip()
cfg = {
    "catalog": dbutils.widgets.get("catalog").strip(),
    "schema": dbutils.widgets.get("schema").strip(),
    "tables": [t.strip() for t in dbutils.widgets.get("tables").split(",") if t.strip()],
    "run_date": datetime.strptime(_run_date, "%Y-%m-%d").date() if _run_date else date.today(),
    "weeks_back": int(dbutils.widgets.get("weeks_back")),
    "detail_weeks": int(dbutils.widgets.get("detail_weeks")),
    "detail_limit": int(dbutils.widgets.get("detail_limit")),
    "top_senders": int(dbutils.widgets.get("top_senders")),
}
assert cfg["catalog"] and cfg["schema"] and cfg["tables"], "catalog / schema / tables 不可為空"
assert cfg["weeks_back"] >= 0 and cfg["detail_weeks"] >= 1, "weeks_back >= 0、detail_weeks >= 1"
print(cfg)

In [ ]:
# [c02] imports
# 本 cell 與 [c03] 不碰 spark / dbutils，可被 tests/ 載入。
from datetime import date, timedelta
from functools import reduce

from pyspark.sql import DataFrame
from pyspark.sql import functions as F

EMPTY = "(空白)"
EMAIL_RE = r"([A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+)"

In [ ]:
# [c03] pure_helpers
def week_range(anchor: date, weeks_back: int) -> list[date]:
    """anchor 所在週（週一起算）加上往回 weeks_back 個完整週，回傳各週週一，舊 → 新。"""
    this_monday = anchor - timedelta(days=anchor.weekday())
    return [this_monday - timedelta(weeks=i) for i in range(weeks_back, -1, -1)]


def week_label(week_start: date, anchor: date) -> str:
    """'9/15 ~ 9/21'；anchor 所在週顯示到 anchor 為止並標 (ongoing)。"""
    week_end = week_start + timedelta(days=6)
    if week_start <= anchor <= week_end:
        return f"{week_start.month}/{week_start.day} ~ {anchor.month}/{anchor.day} (ongoing)"
    return f"{week_start.month}/{week_start.day} ~ {week_end.month}/{week_end.day}"


def _blank_to(col, fill: str = EMPTY):
    c = F.trim(col)
    return F.when(c.isNull() | (c == ""), F.lit(fill)).otherwise(c)


def normalize_mail(df: DataFrame) -> DataFrame:
    """輸入 source / create_date / mail_from / mail_subject；加 week_start 與歸戶鍵 sender_key。

    sender_key = mail_from 內第一個 email（小寫）；抓不到就用整串 mail_from（小寫），空值為 (空白)。
    """
    mail_from = _blank_to(F.col("mail_from"))
    email = F.lower(F.regexp_extract(mail_from, EMAIL_RE, 1))
    return df.select(
        "source",
        F.to_date(F.date_trunc("week", F.col("create_date"))).alias("week_start"),
        "create_date",
        mail_from.alias("mail_from"),
        F.when(email != "", email).otherwise(F.lower(mail_from)).alias("sender_key"),
        _blank_to(F.col("mail_subject")).alias("mail_subject"),
    )


def weekly_counts(df: DataFrame, grid: DataFrame) -> DataFrame:
    """grid 是 (source, week_start) 全組合；每格算筆數與寄件人數，沒資料補 0。"""
    cnt = df.groupBy("source", "week_start").agg(
        F.count(F.lit(1)).alias("record_count"),
        F.countDistinct("sender_key").alias("sender_count"),
    )
    return (
        grid.join(cnt, ["source", "week_start"], "left")
        .fillna(0, ["record_count", "sender_count"])
        .orderBy("source", F.col("week_start").desc())
    )


def sender_summary(df: DataFrame) -> DataFrame:
    """每張表、每週、每位寄件人：幾封、第一封 / 最後一封時間。mail_from 取一個代表寫法。"""
    return (
        df.groupBy("source", "week_start", "sender_key")
        .agg(
            F.count(F.lit(1)).alias("mail_count"),
            F.max("mail_from").alias("mail_from"),
            F.min("create_date").alias("first_at"),
            F.max("create_date").alias("last_at"),
        )
        .orderBy("source", F.col("week_start").desc(), F.col("mail_count").desc(), "sender_key")
    )


def sender_week_matrix(df: DataFrame, weeks: list[date]) -> DataFrame:
    """寄件人 × 週次 筆數對照表（欄名 MM/dd = 該週週一），加 total 與有記錄的週數 active_weeks。"""
    labels = [w.strftime("%m/%d") for w in weeks]
    d = df.withColumn("wk", F.date_format("week_start", "MM/dd"))
    names = d.groupBy("source", "sender_key").agg(F.max("mail_from").alias("mail_from"))
    pv = d.groupBy("source", "sender_key").pivot("wk", labels).count()
    wk_cols = [F.coalesce(F.col(f"`{c}`"), F.lit(0)) for c in labels]
    return (
        names.join(pv, ["source", "sender_key"])
        .select(
            "source",
            "sender_key",
            "mail_from",
            *[c.alias(n) for c, n in zip(wk_cols, labels, strict=True)],
            reduce(lambda a, b: a + b, wk_cols).alias("total"),
            reduce(lambda a, b: a + b, [F.when(c > 0, 1).otherwise(0) for c in wk_cols]).alias(
                "active_weeks"
            ),
        )
        .orderBy("source", F.col("total").desc(), "sender_key")
    )


def mail_detail(df: DataFrame) -> DataFrame:
    """信件明細：新 → 舊。"""
    return df.select("source", "week_start", "create_date", "mail_from", "mail_subject").orderBy(
        "source", F.col("create_date").desc()
    )


def weekly_report(counts: list[dict], senders: list[dict], anchor: date, top_n: int = 20) -> str:
    """把 [c11] / [c12] collect 回來的小結果排成文字報告（新週在上）。"""
    lines = []
    for src in sorted({r["source"] for r in counts}):
        lines.append(f"===== {src} =====")
        rows = [r for r in counts if r["source"] == src]
        rows.sort(key=lambda r: r["week_start"], reverse=True)
        for c in rows:
            label = week_label(c["week_start"], anchor)
            lines.append(f"{label}: {c['record_count']} 筆 / {c['sender_count']} 位寄件人")
            ss = sorted(
                (s for s in senders if s["source"] == src and s["week_start"] == c["week_start"]),
                key=lambda s: (-s["mail_count"], s["sender_key"]),
            )
            lines += [f"    {s['mail_count']:>4}  {s['mail_from']}" for s in ss[:top_n]]
            if len(ss) > top_n:
                lines.append(f"    …另 {len(ss) - top_n} 位")
    return "\n".join(lines)

In [ ]:
# [c10] load_source
# 先用日期過濾、只取 4 個欄位再 union；範圍 = 第一週週一 ~ 本週週日。
weeks = week_range(cfg["run_date"], cfg["weeks_back"])
start_date, end_date = weeks[0], weeks[-1] + timedelta(days=7)  # [start, end)

parts = [
    spark.table(f"{cfg['catalog']}.{cfg['schema']}.{t}")
    .where((F.col("create_date") >= F.lit(start_date)) & (F.col("create_date") < F.lit(end_date)))
    .select(F.lit(t).alias("source"), "create_date", "mail_from", "mail_subject")
    for t in cfg["tables"]
]
# [c11]～[c15] 對同一份資料做多次 action，範圍只有幾週，cache 後在 [c90] unpersist。
df_mail = normalize_mail(reduce(DataFrame.unionByName, parts)).cache()
print(f"range: {start_date} ~ {end_date - timedelta(days=1)}, tables={cfg['tables']}")

In [ ]:
# [c11] weekly_counts
grid = spark.createDataFrame(
    [(t, w) for t in cfg["tables"] for w in weeks], "source string, week_start date"
)
df_counts = weekly_counts(df_mail, grid)
counts_rows = [r.asDict() for r in df_counts.collect()]  # 表數 × 週數 筆，很小
display(df_counts)

In [ ]:
# [c12] sender_summary
df_senders = sender_summary(df_mail)
sender_rows = [r.asDict() for r in df_senders.collect()]  # 已彙總：表 × 週 × 寄件人，量級數百筆
print(weekly_report(counts_rows, sender_rows, cfg["run_date"], cfg["top_senders"]))
display(df_senders)

In [ ]:
# [c13] sender_week_matrix
# 欄名 MM/dd = 該週週一；active_weeks 小於週數的人 = 有幾週沒記。
display(sender_week_matrix(df_mail, weeks))

In [ ]:
# [c14] chart
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(cfg["tables"]), figsize=(7 * len(cfg["tables"]), 5), squeeze=False)
for ax, t in zip(axes[0], cfg["tables"], strict=True):
    rows = sorted((r for r in counts_rows if r["source"] == t), key=lambda r: r["week_start"])
    xs = [week_label(r["week_start"], cfg["run_date"]) for r in rows]
    ys = [r["record_count"] for r in rows]
    bars = ax.bar(xs, ys, color="#4A90D9", edgecolor="white")
    for bar, r in zip(bars, rows, strict=True):
        text = f"{r['record_count']}\n({r['sender_count']} senders)"
        x = bar.get_x() + bar.get_width() / 2
        ax.text(x, bar.get_height(), text, ha="center", va="bottom", fontsize=10)
    ax.set_title(t, fontsize=13, fontweight="bold", pad=10)
    ax.set_ylabel("record count")
    ax.set_ylim(0, max(max(ys, default=0) * 1.35, 1))
    ax.tick_params(axis="x", rotation=25, labelsize=9)
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("Meeting Record Weekly Ingestion Report", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# [c15] mail_detail
# 最近 detail_weeks 週的逐封明細（寄件人、標題），最多 detail_limit 筆。
detail_weeks = weeks[-cfg["detail_weeks"]:]
display(mail_detail(df_mail.where(F.col("week_start").isin(detail_weeks))).limit(cfg["detail_limit"]))

In [ ]:
# [c90] cleanup
df_mail.unpersist()